### Machine Project 1 - Affine Transformations
**IMAGPRO - S11**

*Group Members:*
- Agustines, Alfred Bastin
- De Leon, Allan David
- Depasucat, Michael Angelo
- Marquez, Vincent Alvin
- Padilla, Kai Hiori

---

#### **Import Dependencies and Configurations**


In [23]:
import cv2
import numpy as np
import os
import random
from glob import glob

# ==============================
# Local Directory Config
# ==============================
# input_path: folder containing the original flower images
# output_dir: folder where resized 100x100 images will be saved
input_path = 'D:/Documents/Term 2/IMAGPRO/IMAGPRO_MP1_AffineTransformations/dataset'
output_dir = 'D:/Documents/Term 2/IMAGPRO/IMAGPRO_MP1_AffineTransformations/transformed_dataset'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# base_aug_dir: parent folder for augmented outputs
# subfolders: one subdirectory per transformation type
base_aug_dir = 'D:/Documents/Term 2/IMAGPRO/IMAGPRO_MP1_AffineTransformations/augmented_dataset'
subfolders = ['patch', 'shift', 'rotate', 'flip']
for folder in subfolders:
    path = os.path.join(base_aug_dir, folder)
    if not os.path.exists(path):
        os.makedirs(path)


# ==============================
# Function parameters
# ==============================
# Black patch size (width, height)
P_SIZE = (25, 25) 

# Shift offsets
S_TX = 20
S_TY = 20

# Rotation (degrees)
ROT_A = 30
ROT_B = 60

# Flip modes ('horizontal' or 'vertical')
FLIP_A = 'horizontal'
FLIP_B = 'vertical'

#### **Data Formatting**
Given the image dataset:
- Reshape the images to (100,100,3)
- Save the transformed images as JPEG files in a separate directory


In [24]:
# Collect image files from the input folder using common image extensions
image_extensions = ['*.jfif', '*.jpg', '*.jpeg', '*.png']
image_files = []
for ext in image_extensions:
    image_files.extend(glob(os.path.join(input_path, ext)))
image_files = sorted(image_files)

formatted_images = []
failed_images = []

for fpath in image_files:
    img = cv2.imread(fpath)
    if img is not None:
        resized = cv2.resize(img, (100, 100))
        # Convert all to .jpg format
        original_name = os.path.splitext(os.path.basename(fpath))[0] + '.jpg'
        cv2.imwrite(os.path.join(output_dir, original_name), resized)
        
        formatted_images.append(resized)
    else:
        failed_images.append(os.path.basename(fpath))

# Summary report for data formatting stage
print(f"Successfully formatted {len(formatted_images)} images.")
if failed_images:
    print(f"Failed to read {len(failed_images)} images:")
    for fname in failed_images:
        print(f"  - {fname}")

Successfully formatted 200 images.


#### **Individual Parametrized Functions**

The cell below contains four functions that performs different image transformations:
- ***apply_black_patch***: Randomly put a black patch over a portion of the image
- ***shift_image***: Shift an image sidewards or upwards
- ***rotate_image***: Rotate an image either for 30 or 60 degrees
- ***flip_image***: Flip an image either vertically or horizontally.


In [25]:
# Randomized Black Patch
def apply_black_patch(image, patch_size=(20, 20)):
    # Get image height and width
    h, w = image.shape[:2]
    # Compute random coordinates while keeping the patch inside image bounds
    x = random.randint(0, max(0, w - patch_size[0]))
    y = random.randint(0, max(0, h - patch_size[1]))
    
    patched_img = image.copy()
    # Apply black patch (0 is black in BGR)
    patched_img[y:y+patch_size[1], x:x+patch_size[0]] = 0
    return patched_img

# Image Shift Sidewards (tx) or Upwards (ty)
def shift_image(image, tx, ty):
    h, w = image.shape[:2]
    # Affine translation matrix
    M = np.float32([[1, 0, tx], [0, 1, ty]])
    return cv2.warpAffine(image, M, (w, h))

# Rotate Image by 30 or 60 degrees
def rotate_image(image, angle):
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    # Build 2D rotation matrix with scale = 1.0
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, M, (w, h))

# Flip Image Vertically or Horizontally
def flip_image(image, mode):
    # Flip code: 1 for horizontal, 0 for vertical
    flip_code = 1 if mode == 'horizontal' else 0
    return cv2.flip(image, flip_code)

#### **Execution Proper**

The code segment below runs all four parametrized functions from the cell above, and puts them in their own respective directories.


In [26]:
# Load all formatted (100x100, .jpg) images created in the previous step
transformed_files = sorted(glob(os.path.join(output_dir, '*.jpg')))

if not transformed_files:
    print("No images found in transformed_dataset.")
else:
    print(f"Processing {len(transformed_files)} images...")

    for fpath in transformed_files:
        img = cv2.imread(fpath)
        raw_name = os.path.splitext(os.path.basename(fpath))[0]

        # Randomized Black Patch
        res_patch = apply_black_patch(img, patch_size=P_SIZE)
        cv2.imwrite(os.path.join(base_aug_dir, 'patch', f'patch_{raw_name}.jpg'), res_patch)

        # Image Shift Sidewards (tx) or Upwards (ty)
        res_shift = shift_image(img, tx=S_TX, ty=S_TY)
        cv2.imwrite(os.path.join(base_aug_dir, 'shift', f'shift_{raw_name}.jpg'), res_shift)
        
        # Rotate Image by 30 or 60 degrees
        # Options: ROT_A for 30, ROT_B for 60
        res_rot = rotate_image(img, angle=ROT_A)
        cv2.imwrite(os.path.join(base_aug_dir, 'rotate', f'rot_{raw_name}.jpg'), res_rot)

        # Flip Image Vertically or Horizontally
        # Options: FLIP_A for horizontal, FLIP_B for vertical
        res_flip = flip_image(img, mode=FLIP_A)    

        cv2.imwrite(os.path.join(base_aug_dir, 'flip', f'flip_{raw_name}.jpg'), res_flip)
    print(f"Transformed images in {base_aug_dir}")

Processing 200 images...
Transformed images in D:/Documents/Term 2/IMAGPRO/IMAGPRO_MP1_AffineTransformations/augmented_dataset


#### **Guide Questions:**

1. Define Data Augmentation and discuss its importance and the importance of understanding digital image processing for such an activity.
- Data augmentation increases thre diversity and count of a training dataset by using geometric and color changes on existing images (Murel & Kavlakoglu, 2024). This method expands the dataset affordably, reduces the risk of model overfitting, and enhances resilience to real-world changes like lighting and camera angles. This notebook includes 4 data augmentation techniques that has multiplied the dataset count by 4 times essentially. This data augmentation helps in preventing model overfitting, where the machine learning model is fits too close to the training data, causing a memorization of the training data instead of understanding its features and characteristics (Murel & Kavlakoglu, 2024). Krizhevsky et al. (2012) explain this by generating random 224×224 crops and horizontal reflections from 256×256 images during training, so the model rarely sees the exact same input twice, which helps reduce overfitting.

- Understanding digital image processing is essential because augmentation is implemented through image operations (interpolation, geometric transforms, pixel remapping, and filtering). If these operations are applied without proper control, they can introduce artifacts (e.g., blurring, aliasing, clipped regions, or unrealistic colors) that change the semantic content of the image. In practice, image-processing knowledge helps us choose valid parameter ranges, preserve class-defining features, and maintain label correctness. This makes the augmented dataset both larger and trustworthy, improving model generalization instead of adding noisy or misleading training samples (Shorten & Khoshgoftaar, 2019; Krizhevsky et al., 2012).

2. What other data augmentation techniques are applicable and not applicable to the dataset you have produced? Why?
- For the mixed flower and cat dataset, useful techniques include brightness and contrast adjustments, moderate color jittering, Gaussian noise, random cropping and zooming, mild elastic deformations, and occlusion methods like cutout and random erasing. These techniques effectively simulate real-world conditions while maintaining key features like petal shapes and fur patterns (Shorten & Khoshgoftaar, 2019; Perez & Wang, 2017). 

- Although changing the color values like the brightness, contrast or saturation is a good strategy, it might hinder the capabilities of the model to identify key features if the main characteristic of an object being defined is its color (Shorten & Khoshgoftaar, 2019). For the flowers dataset, this is one of the hindering factors since many flowers have color as their main identifier for images, as scent and feel cannot be seen from an image. For the cat images on the other hand, heavy distortion or ramoval of areas that contain anatomical features, and synthetic methods that mix categories or species are things that can hinder its correct indentification. Shorten and Khoshgoftaar (2019) mentioned that data augmentation techniques that erase or write over certain portions of the image has a disadvantage of potentially removing key features, hindering its correct classification by the model. This notebook shows one of these techniques, with the technique that randomly adds black patches to the image. 

#### **Sources:**

Murel, J., & Kavlakoglu, E. (2024). What is data augmentation? IBM. https://www.ibm.com/think/topics/data-augmentation

Shorten, C., & Khoshgoftaar, T. M. (2019). A survey on image data augmentation for deep learning. Journal of Big Data, 6(1), 1-48. https://doi.org/10.1186/s40537-019-0197-0

Perez, L., & Wang, J. (2017). The effectiveness of data augmentation in image classification using deep learning. arXiv. https://arxiv.org/abs/1712.04621

Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. NeurIPS 2012. https://papers.nips.cc/paper_files/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html